# NeuroSegment-BraTS: Treinamento do Modelo (Deep Learning)

**Notebook 4:** Modelagem e Experimentação

Nesta etapa, instanciamos a arquitetura da rede neural, definimos os parâmetros de otimização e executamos o ciclo de treinamento.

**Metodologia Experimental:**
*   **Modelo:** U-Net 3D (arquitetura padrão ouro para segmentação médica volumétrica).
*   **Função de Perda:** `DiceCELoss`. Uma combinação da *Cross-Entropy* (para classificação geral) com a *Dice Loss* (para otimizar o ganho sobre o desbalanceamento das classes tumorais).
*   **Otimizador:** Adam, que se ajusta bem a gradientes esparsos.
*   **Gestão de Hardware:** Como o sistema possui restrições de memória (16GB RAM/VRAM), o treinamento será feito em "Patches" (blocos menores de 96x96x96 gerados no *DataLoader*) e a validação será reconstruída via `SlidingWindowInferer`.

## 1. Importações e Inicialização

In [ ]:
import optuna
import os
import torch
from monai.networks.nets import UNet
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference
from monai.transforms import AsDiscrete
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Treinamento rodando no dispositivo: {device}")

if device.type == 'cuda':
    torch.cuda.empty_cache()

# Post-processing para validação
post_pred = AsDiscrete(argmax=True, to_onehot=4)
post_label = AsDiscrete(to_onehot=4)

In [ ]:
from tqdm.notebook import tqdm

def train_epoch(model, optimizer, loss_fn, dataloader, device):
    """Treina o modelo por 1 época. Retorna a loss média."""
    model.train()
    epoch_loss = 0
    step = 0
    for batch_data in dataloader:
        step += 1
        inputs = batch_data['image'].to(device)
        labels = batch_data['label'].to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / step

def validate(model, dataloader, device, roi_size=(96, 96, 96)):
    """Avalia o modelo no conjunto de validação. Retorna o Dice médio."""
    model.eval()
    post_pred = AsDiscrete(argmax=True, to_onehot=4)
    post_label = AsDiscrete(to_onehot=4)
    dice = DiceMetric(include_background=False, reduction='mean')
    with torch.no_grad():
        for val_data in dataloader:
            val_inputs = val_data['image'].to(device)
            val_labels = val_data['label'].to(device)
            val_outputs = sliding_window_inference(
                inputs=val_inputs, roi_size=roi_size, sw_batch_size=4, predictor=model
            )
            val_outputs_list = [post_pred(i) for i in val_outputs]
            val_labels_list = [post_label(i) for i in val_labels]
            dice(y_pred=val_outputs_list, y=val_labels_list)
    result = dice.aggregate().item()
    dice.reset()
    return result

## 2. Ingestão do Pipeline de Dados
Abaixo, conectamos este notebook aos iteradores criados no Notebook 3.

In [ ]:
import sys
sys.path.append('../')
from src.data_pipeline import get_dataloaders

# 1. Ingestão do Pipeline
train_loader, val_loader = get_dataloaders()
print("DataLoaders carregados com sucesso no Notebook 4!")

# Validando a conexão (assumindo que train_loader e val_loader estão definidos no kernel)
test_batch = next(iter(train_loader))
print(f"Shape de entrada da imagem: {test_batch['image'].shape}") 
# Esperado: (Batch, Canais, X, Y, Z) -> ex: (2, 4, 96, 96, 96)

## 3. Definição da Arquitetura (U-Net 3D) e Hiperparâmetros
Configuramos a U-Net para aceitar 4 canais de entrada (FLAIR, T1, T1ce, T2) e devolver predições para as nossas 4 classes de interesse (0, 1, 2, 4).

In [ ]:
# 1. O Modelo: U-Net Tridimensional
model = UNet(
    spatial_dims=3,             # Imagem 3D
    in_channels=4,              # 4 Modalidades (FLAIR, T1, T1ce, T2)
    out_channels=4,             # 4 Classes de saída (Fundo/Normal, NCR, ED, ET)
    channels=(16, 32, 64, 128, 256), # Número de filtros por camada (mantido leve por causa dos 16GB de VRAM)
    strides=(2, 2, 2, 2),
    num_res_units=2,
).to(device)

# 2. A Função de Perda (Loss Function)
# Combina Softmax (CrossEntropy) com a métrica espacial (Dice)
loss_function = DiceCELoss(to_onehot_y=True, softmax=True)

# 3. O Otimizador
learning_rate = 1e-4
optimizer = torch.optim.Adam(model.parameters(), learning_rate)

# 4. A Métrica de Avaliação Oficial (Dice Score)
# Mede a sobreposição entre a máscara da IA e o gabarito do médico
dice_metric = DiceMetric(include_background=False, reduction="mean")

## 4. Otimização de Hiperparâmetros com Optuna
Usamos o Optuna para buscar os melhores hiperparâmetros. Cada trial treina por 10 épocas e avalia o Dice na validação.

**Hiperparâmetros buscados (U-Net 3D):**
- `learning_rate`: taxa de aprendizado (escala logarítmica entre 1e-5 e 1e-3)
- `num_res_units`: número de residual units por camada (1, 2 ou 3) — afeta a profundidade da rede

In [ ]:
import optuna

def objective_unet(trial):
    lr = trial.suggest_float('learning_rate', 1e-5, 1e-3, log=True)
    num_res = trial.suggest_int('num_res_units', 1, 3)

    model_trial = UNet(
        spatial_dims=3, in_channels=4, out_channels=4,
        channels=(16, 32, 64, 128, 256), strides=(2, 2, 2, 2), num_res_units=num_res,
    ).to(device)

    optimizer_trial = torch.optim.Adam(model_trial.parameters(), lr=lr)
    loss_fn = DiceCELoss(to_onehot_y=True, softmax=True)

    # DataLoader com num_workers=0 para evitar deadlock no Jupyter
    from monai.data import DataLoader as _MDL
    train_loader_opt = _MDL(train_loader.dataset, batch_size=1, shuffle=True, num_workers=0)
    val_loader_opt = _MDL(val_loader.dataset, batch_size=1, num_workers=0)

    for epoch in range(10):
        loss = train_epoch(model_trial, optimizer_trial, loss_fn, train_loader_opt, device)
        print(f'  Trial {trial.number} | Época {epoch+1}/10 | Loss: {loss:.4f}')

    result = validate(model_trial, val_loader_opt, device)

    del model_trial, optimizer_trial
    del train_loader_opt, val_loader_opt
    torch.cuda.empty_cache()

    return result

study = optuna.create_study(direction='maximize')
study.optimize(objective_unet, n_trials=10, show_progress_bar=True)

In [ ]:
import optuna.visualization.matplotlib as vis_plot

ax = vis_plot.plot_optimization_history(study)
ax.set_title('U-Net 3D — Histórico de Otimização')
plt.tight_layout()
plt.show()

ax2 = vis_plot.plot_param_importances(study)
plt.tight_layout()
plt.show()

print('=' * 50)
print('Melhores hiperparâmetros (U-Net 3D):')
print('=' * 50)
for key, value in study.best_params.items():
    if isinstance(value, float):
        print(f'  {key}: {value:.6f}')
    else:
        print(f'  {key}: {value}')
print(f'  Dice Score: {study.best_value:.4f}')
print('=' * 50)

In [ ]:
# Redefinir modelo e otimizador com os melhores hiperparâmetros
learning_rate = study.best_params['learning_rate']
num_res_units = study.best_params['num_res_units']

model = UNet(
    spatial_dims=3, in_channels=4, out_channels=4,
    channels=(16, 32, 64, 128, 256), strides=(2, 2, 2, 2), num_res_units=num_res_units,
).to(device)

loss_function = DiceCELoss(to_onehot_y=True, softmax=True)
optimizer = torch.optim.Adam(model.parameters(), learning_rate)
dice_metric = DiceMetric(include_background=False, reduction='mean')

print(f'U-Net 3D redefinida com lr={learning_rate:.6f}, num_res_units={num_res_units}')

## 4. O Ciclo de Treinamento e Validação
Executamos o treinamento iterativo. A cada *batch*, o modelo tenta segmentar o tumor, calcula o erro e ajusta seus pesos. Ao final de cada época, rodamos uma etapa de validação (sem aprender) para medir a performance real.

In [ ]:
from tqdm.notebook import tqdm
# Configurações do Loop
max_epochs = 100
val_interval = 4

best_metric = -1
best_metric_epoch = -1
epoch_loss_values = []
metric_values = []

print('INICIANDO O TREINAMENTO...')

for epoch in tqdm(range(max_epochs), desc='Treinamento U-Net 3D'):
    # === MODO DE TREINO ===
    epoch_loss = train_epoch(model, optimizer, loss_function, train_loader, device)
    epoch_loss_values.append(epoch_loss)

    # === MODO DE VALIDAÇÃO ===
    if (epoch + 1) % val_interval == 0:
        metric = validate(model, val_loader, device)
        metric_values.append(metric)
        print(f'\n>>> Época {epoch+1}/{max_epochs} | Loss: {epoch_loss:.4f} | Dice: {metric:.4f}')

        if metric > best_metric:
            best_metric = metric
            best_metric_epoch = epoch + 1
            torch.save(model.state_dict(), os.path.join('../models', 'best_metric_model_Unet_3D.pth'))
            print('Novo recorde! Modelo salvo no disco.')

print(f'\nTREINAMENTO CONCLUÍDO! Melhor Dice: {best_metric:.4f} na época {best_metric_epoch}')

## 5. Curvas de Aprendizado
Visualização da evolução do loss de treino e do Dice de validação ao longo das épocas.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(epoch_loss_values, label='Train Loss', color='tab:red')
ax1.set_xlabel('Época')
ax1.set_ylabel('Loss')
ax1.set_title('Loss de Treino por Época')
ax1.legend()

val_epochs = [i * val_interval for i in range(1, len(metric_values) + 1)]
ax2.plot(val_epochs, metric_values, label='Val Dice', color='tab:green')
ax2.set_xlabel('Época')
ax2.set_ylabel('Dice Score')
ax2.set_title('Dice de Validação')
ax2.legend()

plt.suptitle('U-Net 3D — Curvas de Aprendizado', fontsize=14)
plt.tight_layout()
plt.savefig('../models/unet3d_learning_curves.png', dpi=150)
plt.show()

## 6. Avaliação Detalhada do Melhor Modelo
Carregamos o melhor checkpoint e calculamos métricas detalhadas por classe na base de validação:
- **Dice Score** por classe (NCR, ED, ET)
- **Hausdorff Distance 95** por classe (métrica oficial BraTS)
- **Sensitivity** e **Specificity** por classe

| Métrica | O que mede | Interpretação |
|---------|-----------|---------------|
| **Dice** | Sobreposição entre predição e gabarito | 0–1, maior = melhor |
| **HD95** | Hausdorff Distance 95th percentile — pior distância de fronteira | mm, menor = melhor |
| **Sensitivity** | % de voxels tumorais corretamente detectados | 0–1, maior = melhor |
| **Specificity** | % de voxels saudáveis corretamente identificados | 0–1, maior = melhor |


In [ ]:
from monai.metrics import DiceMetric, HausdorffDistanceMetric, ConfusionMatrixMetric

# Carregar o melhor modelo salvo
best_model = UNet(
    spatial_dims=3,
    in_channels=4,
    out_channels=4,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
).to(device)
best_model.load_state_dict(torch.load(os.path.join('../models', 'best_metric_model_Unet_3D.pth'), weights_only=True))
print('Modelo U-Net 3D carregado com sucesso!')
best_model.eval()

# Métricas por classe
dice_per_class = DiceMetric(include_background=False, reduction="mean_batch")
hd95_per_class = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean_batch")
sensitivity_metric = ConfusionMatrixMetric(include_background=False, metric_name='sensitivity', reduction='mean_batch')
specificity_metric = ConfusionMatrixMetric(include_background=False, metric_name='specificity', reduction='mean_batch')

post_pred = AsDiscrete(argmax=True, to_onehot=4)
post_label = AsDiscrete(to_onehot=4)

with torch.no_grad():
    for val_data in val_loader:
        val_inputs = val_data['image'].to(device)
        val_labels = val_data['label'].to(device)
        
        val_outputs = sliding_window_inference(
            inputs=val_inputs, roi_size=(128, 128, 128), sw_batch_size=1, predictor=best_model
        )
        
        val_outputs_list = [post_pred(i) for i in val_outputs]
        val_labels_list = [post_label(i) for i in val_labels]
        
        dice_per_class(y_pred=val_outputs_list, y=val_labels_list)
        hd95_per_class(y_pred=val_outputs_list, y=val_labels_list)
        sensitivity_metric(y_pred=val_outputs_list, y=val_labels_list)
        specificity_metric(y_pred=val_outputs_list, y=val_labels_list)

# Agregar resultados
dice_scores = dice_per_class.aggregate()
hd95_scores = hd95_per_class.aggregate()
sens_raw = sensitivity_metric.aggregate()
spec_raw = specificity_metric.aggregate()

# ConfusionMatrixMetric com mean_batch retorna lista de tensores (cada um com C elementos)
# Flatten para 1D: ex: [tensor([a,b,c])] -> tensor([a,b,c])
def to_1d(x):
    if isinstance(x, list):
        x = torch.cat([t.flatten() for t in x])
    if isinstance(x, torch.Tensor) and x.dim() > 1:
        x = x.flatten()
    return x

sens_scores = to_1d(sens_raw)
spec_scores = to_1d(spec_raw)

class_names = ['NCR (TC)', 'ED (WT)', 'ET']
header = '{:<12} {:>8} {:>10} {:>8} {:>8}'.format('Classe', 'Dice', 'HD95', 'Sens', 'Spec')
print(header)
print('-' * 50)
for i, name in enumerate(class_names):
    d = dice_scores[i].item() if hasattr(dice_scores[i], 'item') else float(dice_scores[i])
    h = hd95_scores[i].item() if hasattr(hd95_scores[i], 'item') else float(hd95_scores[i])
    s = sens_scores[i].item() if i < len(sens_scores) else 0.0
    sp = spec_scores[i].item() if i < len(spec_scores) else 0.0
    print('{:<12} {:>8.4f} {:>10.2f} {:>8.4f} {:>8.4f}'.format(name, d, h, s, sp))
print('-' * 50)
mean_dice = dice_scores.mean().item() if hasattr(dice_scores, 'mean') else float(dice_scores)
mean_hd95 = hd95_scores.mean().item() if hasattr(hd95_scores, 'mean') else float(hd95_scores)
print('{:<12} {:>8.4f} {:>10.2f}'.format('Média', mean_dice, mean_hd95))